In [14]:
import argparse
import os
import pickle
import sys
import typing

import pandas as pd
import torch
from Bio import SeqIO
from typing import List, Union, Optional, Callable, Sequence
from transformers import (
    EsmForMaskedLM, 
    PretrainedConfig, 
    EsmTokenizer, 
    DataCollatorForLanguageModeling, 
    Trainer
)

from tokenizers import Tokenizer
import torch
import torch.nn.functional as F

import einops
import yaml
import sys
import json
import functools
import os

import numpy as np
from huggingface_hub import hf_hub_download
from datasets import Dataset
import math

In [2]:
import transformer_lens
import transformer_lens.utils as utils
from transformer_lens.hook_points import (
    HookedRootModule,
    HookPoint,
)

# Hooking utilities
from transformer_lens import (
    HookedTransformer,
    HookedTransformerConfig,
    FactoredMatrix,
    ActivationCache,
)

sys.path.append("../../scripts")
from compute_node_embeddings import load_sequences, get_protein_sequence
from interp_utils import get_hooked_state_dict, get_hooked_esm_config, get_logits_hooked_esm

In [3]:
device = "cuda"
CONTEXT_LEN = 1000

In [4]:
# ESM-2 config
with open("./esm_config35M.json", "r") as file:
    esm_config = json.load(file)
model_name = esm_config["_name_or_path"]
model_name = model_name[model_name.find("facebook"):]
esm_config["token_dropout"] = False
esm_config["model_name"] = model_name
esm_config = PretrainedConfig(**esm_config)

# ESM-2 tokenizezr config
REPO_ID = esm_config.model_name
vocab_file = "vocab.txt"
special_tokens_map_file = "special_tokens_map.json"
tokenizer_config = {}
tokenizer_config["vocab_file"] = hf_hub_download(repo_id=REPO_ID, filename=vocab_file)
tokenizer_config["model_max_length"] = CONTEXT_LEN
with open(hf_hub_download(repo_id=REPO_ID, filename=special_tokens_map_file), "r") as f:
    tokenizer_config = {**tokenizer_config, **(json.load(f))}

# hooked ESM-2 config
hooked_esm_config = get_hooked_esm_config(esm_config, context_len=CONTEXT_LEN)

In [5]:
# ESM-2 35M
model_name = esm_config.model_name
tokenizer = tokenizer = EsmTokenizer(**tokenizer_config)
reg_esm = EsmForMaskedLM(esm_config).to(device)

# hooked ESM-2
hooked_esm = HookedTransformer(hooked_esm_config)
print(hooked_esm.load_state_dict(get_hooked_state_dict(reg_esm.state_dict(), hooked_esm_config)))

# helper function to get logits from hooked ESM-2 head
apply_esm_lm_head = functools.partial(get_logits_hooked_esm, ESM2_lm_head=reg_esm.lm_head)
print(model_name)

<All keys matched successfully>
facebook/esm2_t12_35M_UR50D


<img src="./ESM2_architecture.png" width="1000"/>

In [10]:
# data loading
with open("../../config/pathogen_config.yaml", "r") as config_file:
    config = yaml.safe_load(config_file)
    
pathogens = list(config["pathogens"].keys())
pathogen_idx = 7
print(pathogens[pathogen_idx])
fasta_file = f"../../data/pathogen/{pathogens[pathogen_idx]}/alignment.fasta"
data = load_sequences(fasta_file)
sequence_names, sequences = list(zip(*list(data.items())))

influenza_h1n1pdm_ha


In [ ]:
N = 0
test_seq = np.unique(sequences)[50]
# test_seq = np.unique(sequences)[1000]
print(test_seq)
tokenized_seq = tokenizer(test_seq, return_tensors="pt", padding=False).input_ids[:,:hooked_esm_config.n_ctx].to(device)
print(tokenized_seq.shape)

In [ ]:
print(tokenized_seq)

In [ ]:
# run respective models
with torch.no_grad():
    outputs = reg_esm.forward(tokenized_seq, output_hidden_states=True)
    reg_esm_hs = outputs.hidden_states

hooked_toks, cache = hooked_esm.run_with_cache(tokenized_seq)

In [ ]:
for l in range(esm_config.num_hidden_layers + 2):
    # 0th layer is embedding layer
    if l < esm_config.num_hidden_layers:
        actual_output = reg_esm_hs[l]
        hooked_output = cache[f"blocks.{l}.hook_resid_pre"]
    
    # final layer
    elif l == esm_config.num_hidden_layers:
        actual_output = reg_esm_hs[l]
        hooked_output = cache[f"ln_final.hook_normalized"]

    # ESM LM head:
    else:
        actual_output = outputs.logits
        with torch.no_grad():
            hooked_output = apply_esm_lm_head(hooked_toks)
    
    is_close = torch.all(torch.isclose(actual_output, hooked_output, rtol=0.1)).item()
    print(f"layer {l}: {is_close}")

In [ ]:
print(outputs.logits[0])

## Perplexity of ESM on different pathogens

In [11]:
# data loading
with open("../../config/pathogen_config.yaml", "r") as config_file:
    config = yaml.safe_load(config_file)
pathogens = list(config["pathogens"].keys())

def tokenizer_for_map(seq): #Tokenizer and params including special_tokens_mask required for MLM
    return tokenizer(
        seq["input_ids"],
        return_tensors="pt", 
        return_special_tokens_mask=True,
        truncation=True,
        padding="max_length",
        max_length=1000,
    )

In [17]:
# small thing to turn off annoying wand questions
os.environ["WANDB_DISABLED"] = "true"

for pathogen_name in pathogens:
    fasta_file = f"../../data/pathogen/{pathogen_name}/alignment.fasta"
    data = load_sequences(fasta_file)
    sequence_names, sequences = list(zip(*list(data.items())))
    uniq_seqs, unique_inds = np.unique(sequences, return_index=True) # For the purpose of eval, I only care about unique sequences 


    # identical code to how it's compute_node_embeddings.py
    protein_coords = config["pathogens"][pathogen_name]["protein_coords"]
    cut_seqs = [get_protein_sequence(x, protein_coords) for x in uniq_seqs]
    cut_seqs = list(np.unique([x for x in cut_seqs if len(x) > 3])) # remove any identical sequences again in case more appear after trimming

    # create dataset
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer,return_tensors='pt',mlm_probability=0.15)
    eval_dict = {"input_ids":list(cut_seqs)}

    eval_dset = Dataset.from_dict(eval_dict)

    column_names = eval_dset.column_names #This will be the names of all the old columns, to then be deleted after the new tokenized columns are added.
    eval_dataset = eval_dset.map( 
        tokenizer_for_map,
        batched=True,
        num_proc=8,
        remove_columns=column_names,
    )

    # evaluate ESM-2 model on dataset 
    evaluator = Trainer(
      model=reg_esm,
      data_collator=data_collator,
      eval_dataset=eval_dataset,
    )
    eval_results = evaluator.evaluate()
    print(f"Pathogen: {pathogen_name}\nESM-2 Perplexity: {math.exp(eval_results['eval_loss']):.4f}\n")

Map (num_proc=8): 100%|██████████████████████████████████████████████████████████████████████| 971/971 [00:00<00:00, 1310.39 examples/s]


{'eval_loss': 3.5707337856292725, 'eval_model_preparation_time': 0.0044, 'eval_runtime': 8.9063, 'eval_samples_per_second': 109.024, 'eval_steps_per_second': 13.698}
Pathogen: influenza_h3n2_ha
ESM-2 Perplexity: 35.5427



Map (num_proc=8): 100%|████████████████████████████████████████████████████████████████████| 1658/1658 [00:00<00:00, 1894.26 examples/s]


{'eval_loss': 3.617932081222534, 'eval_model_preparation_time': 0.0018, 'eval_runtime': 15.1724, 'eval_samples_per_second': 109.277, 'eval_steps_per_second': 13.709}
Pathogen: rsv_a_g
ESM-2 Perplexity: 37.2604



Map (num_proc=8): 100%|████████████████████████████████████████████████████████████████████| 1699/1699 [00:01<00:00, 1286.41 examples/s]


...

{'eval_loss': 3.5883727073669434, 'eval_model_preparation_time': 0.0045, 'eval_runtime': 1.4634, 'eval_samples_per_second': 107.967, 'eval_steps_per_second': 13.667}
Pathogen: rsv_b_g
ESM-2 Perplexity: 36.1752



Map (num_proc=8): 100%|███████████████████████████████████████████████████████████████████████| 186/186 [00:00<00:00, 327.81 examples/s]


{'eval_loss': 3.552466630935669, 'eval_model_preparation_time': 0.0044, 'eval_runtime': 1.72, 'eval_samples_per_second': 108.137, 'eval_steps_per_second': 13.953}
Pathogen: oc43_a_s1
ESM-2 Perplexity: 34.8993



Map (num_proc=8): 100%|█████████████████████████████████████████████████████████████████████████| 82/82 [00:00<00:00, 151.57 examples/s]


{'eval_loss': 3.568345308303833, 'eval_model_preparation_time': 0.0044, 'eval_runtime': 0.7712, 'eval_samples_per_second': 106.325, 'eval_steps_per_second': 14.263}
Pathogen: cov_229e_s1
ESM-2 Perplexity: 35.4579

